<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_01_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 1: Configuration, Storage Setup, and Raw Data Ingestion

This notebook initializes the reproducible Phase 2 configuration, mounts the
persistent Google Drive storage, and prepares the Data Lake directory structure.
It checks whether Kvasir-Capsule is already available at the configured storage
location and downloads it from the official source only when it is missing.
The dataset contents are then inventoried and validated before downstream
processing begins.

### 1. Install dependencies and imports

In [ ]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    gdown

In [ ]:




from pathlib import Path
from collections import defaultdict

import re
import mimetypes
import os
import json
import random
import warnings
import subprocess
import shutil
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display
from collections import deque
from uuid import uuid4

import filecmp
import lzma
import stat
import tempfile

import cv2
import numpy as np
import pandas as pd
import torch
import zipfile
import tarfile
import zlib
import gzip


from tqdm.auto import tqdm

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Load declarative configuration and reproducibility

In [ ]:
CONFIG = {
    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    "seed": 42,

    # --------------------------------------------------------------
    # Persistent storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Raw-dataset acquisition
    # --------------------------------------------------------------

    "dataset_download_enabled": True,

    "dataset_google_drive_folder_url": (
        "https://drive.google.com/drive/folders/"
        "18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z"
    ),

    "dataset_archive_repair_enabled": True,

    # --------------------------------------------------------------
    # Raw-dataset validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "osf_storage_subdir": (
            "osfstorage"
        ),

        "required_metadata_file": (
            "metadata.csv"
        ),

        "labelled_images_subdir": (
            "labelled_images"
        ),

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },
}


CONFIG

In [ ]:
# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

### 3. Mount Google Drive Storage Backend

In [ ]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

### 4. Define data paths

In [ ]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

### 5. Dataset inventory verification

In [ ]:
def build_raw_dataset_inventory(
    config,
    dirs,
):
    """
    Builds a declarative inventory of the raw
    Kvasir-Capsule dataset.

    The filesystem is inspected but not modified.

    Returns:
        One dictionary containing paths, observed counts,
        expected minimums, and the final validation result.
    """

    required_dir_keys = {
        "raw_data_dir",
        "dataset_root_dir",
        "labelled_images_dir",
        "metadata_path",
    }

    missing_dir_keys = sorted(
        required_dir_keys
        - set(dirs)
    )

    if missing_dir_keys:
        raise KeyError(
            "Dataset verification is missing DIRS keys: "
            f"{missing_dir_keys}"
        )

    validation = config[
        "dataset_validation"
    ]

    required_validation_keys = {
        "minimum_video_files",
        "minimum_labelled_images",
        "image_extensions",
        "video_extensions",
    }

    missing_validation_keys = sorted(
        required_validation_keys
        - set(validation)
    )

    if missing_validation_keys:
        raise KeyError(
            "Dataset verification is missing CONFIG keys: "
            f"{missing_validation_keys}"
        )

    raw_data_dir = Path(
        dirs[
            "raw_data_dir"
        ]
    )

    dataset_root_dir = Path(
        dirs[
            "dataset_root_dir"
        ]
    )

    labelled_images_dir = Path(
        dirs[
            "labelled_images_dir"
        ]
    )

    metadata_path = Path(
        dirs[
            "metadata_path"
        ]
    )

    image_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "image_extensions"
        ]
    )

    video_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "video_extensions"
        ]
    )

    minimum_video_files = int(
        validation[
            "minimum_video_files"
        ]
    )

    minimum_labelled_images = int(
        validation[
            "minimum_labelled_images"
        ]
    )

    if minimum_video_files < 1:
        raise ValueError(
            "minimum_video_files must be positive."
        )

    if minimum_labelled_images < 1:
        raise ValueError(
            "minimum_labelled_images must be positive."
        )

    raw_data_dir_exists = (
        raw_data_dir.is_dir()
    )

    dataset_root_dir_exists = (
        dataset_root_dir.is_dir()
    )

    labelled_images_dir_exists = (
        labelled_images_dir.is_dir()
    )

    metadata_available = (
        metadata_path.is_file()
    )

    video_count = (
        sum(
            1
            for path
            in dataset_root_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in video_extensions
            )
        )
        if dataset_root_dir_exists
        else 0
    )

    labelled_image_count = (
        sum(
            1
            for path
            in labelled_images_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in image_extensions
            )
        )
        if labelled_images_dir_exists
        else 0
    )

    minimum_video_count_met = (
        video_count
        >= minimum_video_files
    )

    minimum_labelled_image_count_met = (
        labelled_image_count
        >= minimum_labelled_images
    )

    validation_passed = all(
        [
            raw_data_dir_exists,
            dataset_root_dir_exists,
            labelled_images_dir_exists,
            metadata_available,
            minimum_video_count_met,
            minimum_labelled_image_count_met,
        ]
    )

    return {
        "raw_data_dir":
            str(raw_data_dir),

        "dataset_root_dir":
            str(dataset_root_dir),

        "labelled_images_dir":
            str(labelled_images_dir),

        "metadata_path":
            str(metadata_path),

        "raw_data_dir_exists":
            raw_data_dir_exists,

        "dataset_root_dir_exists":
            dataset_root_dir_exists,

        "labelled_images_dir_exists":
            labelled_images_dir_exists,

        "metadata_available":
            metadata_available,

        "video_count":
            video_count,

        "minimum_video_files":
            minimum_video_files,

        "minimum_video_count_met":
            minimum_video_count_met,

        "labelled_image_count":
            labelled_image_count,

        "minimum_labelled_images":
            minimum_labelled_images,

        "minimum_labelled_image_count_met":
            minimum_labelled_image_count_met,

        "validation_passed":
            validation_passed,
    }


def verify_raw_dataset(
    config,
    dirs,
):
    """
    Returns True when the raw Kvasir-Capsule dataset
    satisfies all declared minimum requirements.
    """

    inventory = (
        build_raw_dataset_inventory(
            config=config,
            dirs=dirs,
        )
    )

    return bool(
        inventory[
            "validation_passed"
        ]
    )

### 6. Dataset provisioning / archive-recovery

In [ ]:

# ------------------------------------------------------------------
# Expected labelled-image manifest: original thresholds retained
# ------------------------------------------------------------------

EXPECTED_LABELLED_CLASS_COUNTS = {
    "Ampulla of vater": 10,
    "Angiectasia": 866,
    "Blood - fresh": 446,
    "Blood - hematin": 12,
    "Erosion": 506,
    "Erythema": 159,
    "Foreign body": 776,
    "Ileocecal valve": 4189,
    "Lymphangiectasia": 592,
    "Normal clean mucosa": 34338,
    "Polyp": 55,
    "Pylorus": 1529,
    "Reduced mucosal view": 2906,
    "Ulcer": 854,
}

EXPECTED_LABELLED_IMAGE_COUNT = sum(
    EXPECTED_LABELLED_CLASS_COUNTS.values()
)
EXPECTED_LABELLED_CLASS_COUNT = len(EXPECTED_LABELLED_CLASS_COUNTS)

# Retained from the original class scanner.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

dataset_root_dir = Path(DIRS["dataset_root_dir"])
labelled_images_dir = Path(DIRS["labelled_images_dir"])
archive_search_root = Path(DIRS["raw_data_dir"])


# ------------------------------------------------------------------
# Dataset presence and optional download -- NEW
# ------------------------------------------------------------------

def dataset_contains_files(dataset_root):
    """
    Checks for existing files, not dataset completeness.

    A missing directory, an empty directory, or a tree containing
    only empty subdirectories is treated as absent.
    Existing files are left for the inventory and validation steps.
    """
    dataset_root = Path(dataset_root)

    if not dataset_root.exists():
        return False

    if not dataset_root.is_dir():
        raise NotADirectoryError(
            f"Dataset location is not a directory: {dataset_root}"
        )

    return any(path.is_file() for path in dataset_root.rglob("*"))


def run_gdown_download(command):
    """Streams download output and retains recent diagnostics on failure."""
    recent_output = deque(maxlen=60)

    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        try:
            if process.stdout is None:
                raise RuntimeError("Unable to capture gdown output.")

            for line in process.stdout:
                print(line, end="", flush=True)
                recent_output.append(line)

            return_code = process.wait()

        except BaseException:
            # Do not leave the download running after cell interruption.
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise

    if return_code != 0:
        diagnostics = "".join(recent_output).strip()
        raise RuntimeError(
            f"Google Drive download failed with exit code {return_code}.\n"
            f"{diagnostics or 'No diagnostic output was captured.'}\n"
            "Partially downloaded files may remain in the destination. "
            "They have not been validated as a complete dataset."
        )


def ensure_raw_dataset_present(config, dirs):
    """
    Reuses existing dataset files or downloads an absent dataset.

    Returns 'already_present' or 'downloaded'. Neither status means
    that content validation has passed.

    Existing but incomplete contents are handled by the validation
    and archive-recovery steps below, not by a full redownload.
    """
    destination = Path(dirs["dataset_root_dir"])

    if config["storage_backend"] == "google_drive":
        # Matches the /content/drive mount used by mount_storage().
        # Checking directory existence alone could accept local runtime
        # directories created while Google Drive was not mounted.
        drive_mount = Path("/content/drive")

        if not drive_mount.is_mount():
            raise RuntimeError(
                "Google Drive is not mounted at /content/drive. "
                "Run the Google Drive mount cell first."
            )

        if not destination.resolve().is_relative_to(drive_mount.resolve()):
            raise ValueError(
                "The dataset destination is outside the mounted Drive: "
                f"{destination}"
            )

    if dataset_contains_files(destination):
        print(f"Existing dataset files found at: {destination}")
        print("Skipping download; continuing with content validation.")
        return "already_present"

    if not config["dataset_download_enabled"]:
        raise RuntimeError(
            f"No dataset files were found at: {destination}. "
            "Set CONFIG['dataset_download_enabled'] = True "
            "to allow the Google Drive download."
        )

    folder_url = config["dataset_google_drive_folder_url"]
    if not isinstance(folder_url, str) or not folder_url.strip():
        raise ValueError(
            "dataset_google_drive_folder_url must be a non-empty string."
        )

    gdown_executable = shutil.which("gdown")
    if gdown_executable is None:
        raise RuntimeError(
            "gdown is not available on PATH. "
            "Run %pip install -q gdown in the dependencies cell."
        )

    destination.mkdir(parents=True, exist_ok=True)

    command = [
        gdown_executable,
        "--folder",
        folder_url.strip(),
        "-O",
        str(destination),
    ]

    print(f"Downloading Kvasir-Capsule to: {destination}")
    run_gdown_download(command)

    if not dataset_contains_files(destination):
        raise RuntimeError(
            "gdown completed but no dataset files were found at: "
            f"{destination}"
        )

    print("Download completed; continuing with content validation.")
    return "downloaded"


# ------------------------------------------------------------------
# Labelled-image scanner and report
# ------------------------------------------------------------------

def scan_labelled_image_classes(labelled_images_dir, expected_class_counts):
    """Compares the physical class inventory with the declared counts."""
    labelled_images_dir = Path(labelled_images_dir)
    actual_counts = {}

    for class_name in expected_class_counts:
        class_dir = labelled_images_dir / class_name
        actual_counts[class_name] = (
            sum(
                1
                for path in class_dir.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )
            if class_dir.is_dir()
            else 0
        )

    existing_class_dirs = (
        {
            path.name
            for path in labelled_images_dir.iterdir()
            if path.is_dir()
        }
        if labelled_images_dir.is_dir()
        else set()
    )
    expected_classes = set(expected_class_counts)
    missing_classes = [
        name for name in expected_class_counts if actual_counts[name] == 0
    ]
    incomplete_classes = {
        name: {
            "actual": actual_counts[name],
            "expected": expected,
            "missing": expected - actual_counts[name],
        }
        for name, expected in expected_class_counts.items()
        if 0 < actual_counts[name] < expected
    }
    excess_classes = {
        name: {
            "actual": actual_counts[name],
            "expected": expected,
            "excess": actual_counts[name] - expected,
        }
        for name, expected in expected_class_counts.items()
        if actual_counts[name] > expected
    }
    total_actual = sum(actual_counts.values())
    total_expected = sum(expected_class_counts.values())
    expected_classes_present = not missing_classes
    expected_class_counts_met = all(
        actual_counts[name] == expected
        for name, expected in expected_class_counts.items()
    )

    return {
        "actual_counts": actual_counts,
        "total_actual": total_actual,
        "total_expected": total_expected,
        "existing_class_count": len(existing_class_dirs & expected_classes),
        "expected_class_count": len(expected_classes),
        "missing_classes": missing_classes,
        "incomplete_classes": incomplete_classes,
        "excess_classes": excess_classes,
        "unexpected_classes": sorted(existing_class_dirs - expected_classes),
        "expected_classes_present": expected_classes_present,
        "expected_class_counts_met": expected_class_counts_met,
        "expected_total_count_met": total_actual == total_expected,
        "complete": (
            expected_classes_present
            and expected_class_counts_met
            and total_actual == total_expected
        ),
    }


def build_labelled_class_report(scan_report, expected_class_counts):
    """Builds a tabular comparison, with one row per expected class."""
    return (
        pd.Series(expected_class_counts, name="expected_images")
        .rename_axis("class_name")
        .to_frame()
        .join(
            pd.Series(scan_report["actual_counts"], name="actual_images")
            .rename_axis("class_name")
        )
        .assign(
            difference=lambda data: (
                data["actual_images"] - data["expected_images"]
            ),
            status=lambda data: (
                pd.Series("complete", index=data.index, dtype="string")
                .mask(data["actual_images"].gt(data["expected_images"]), "excess")
                .mask(data["actual_images"].lt(data["expected_images"]), "incomplete")
                .mask(data["actual_images"].eq(0), "missing")
            ),
        )
        .reset_index()
        .loc[:, [
            "class_name", "actual_images", "expected_images", "difference", "status"
        ]]
    )


# ------------------------------------------------------------------
# Archive discovery, inspection, and image recovery
# ------------------------------------------------------------------

def normalize_class_key(value):
    """Produces a comparable key for class/archive names."""
    value = re.sub(r"\.(?:tar\.gz|tgz|zip)$", "", value.lower())
    return re.sub(r"[^a-z0-9]+", "", value)


def infer_archive_class(archive_path, expected_classes):
    """Infers a class from an archive filename."""
    archive_key = normalize_class_key(Path(archive_path).name)
    for class_name in sorted(expected_classes):
        if archive_key == normalize_class_key(class_name):
            return class_name
    return None


def find_dataset_archives(search_root):
    """Finds ZIP, TAR.GZ, and TGZ archives recursively."""
    search_root = Path(search_root)
    if not search_root.is_dir():
        return []

    return sorted(
        path
        for path in search_root.rglob("*")
        if path.is_file()
        and path.name.lower().endswith((".zip", ".tar.gz", ".tgz"))
    )


def inspect_archive_for_classes(archive_path, target_classes):
    """Identifies target classes; archive errors propagate to repair."""
    archive_path = Path(archive_path)
    target_classes = set(target_classes)
    found_classes = set()

    archive_class = infer_archive_class(archive_path, target_classes)
    if archive_class is not None:
        return {archive_class}

    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            member_names = [
                member.filename for member in archive.infolist()
                if not member.is_dir()
            ]
    else:
        with tarfile.open(archive_path, "r:*") as archive:
            member_names = [
                member.name for member in archive.getmembers() if member.isfile()
            ]

    normalized_targets = {
        normalize_class_key(name): name for name in target_classes
    }
    for member_name in member_names:
        for part in Path(member_name.replace("\\", "/")).parts:
            key = normalize_class_key(part)
            if key in normalized_targets:
                found_classes.add(normalized_targets[key])

    return found_classes


def recover_required_images_from_archive(
    archive_path, labelled_images_dir, target_classes,
    raw_root, quarantine_dir, replacement_records,
):
    """
    Writes each member to a temporary file before publishing it.

    Existing files identical to the archive are reused. Files with
    different bytes are preserved in quarantine before replacement;
    a mismatch is recorded without claiming the old image is corrupt.
    Source-image decoding/QC remains a separate pipeline stage.
    """
    archive_path = Path(archive_path)
    labelled_images_dir = Path(labelled_images_dir)
    target_classes = set(target_classes)
    recovered_counts = {name: 0 for name in sorted(target_classes)}
    archive_class = infer_archive_class(archive_path, target_classes)
    destinations_seen = set()

    def image_destination(member_name):
        member_path = Path(member_name.replace("\\", "/"))
        if member_path.suffix.lower() not in IMAGE_EXTENSIONS:
            return None, None

        matched_class = archive_class
        if matched_class is None:
            matched_classes = [
                class_name for class_name in sorted(target_classes)
                if any(
                    normalize_class_key(part) == normalize_class_key(class_name)
                    for part in member_path.parts[:-1]
                )
            ]
            if len(matched_classes) > 1:
                raise ValueError(f"Ambiguous archive image class: {member_name}")
            matched_class = matched_classes[0] if matched_classes else None

        if matched_class is None:
            return None, None

        # Keep only the basename; do not extract arbitrary archive paths.
        destination = labelled_images_dir / matched_class / member_path.name
        if not destination.resolve().is_relative_to(labelled_images_dir.resolve()):
            raise ValueError(f"Image destination escapes the labelled directory: {destination}")
        if destination in destinations_seen:
            raise ValueError(f"Archive members collide at the same image path: {destination}")
        destinations_seen.add(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        return matched_class, destination

    def publish_member(source, destination, expected_size):
        with tempfile.NamedTemporaryFile(
            dir=destination.parent, prefix=".extract_", suffix=".part", delete=False
        ) as output:
            temporary_path = Path(output.name)
            try:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            except BaseException:
                output.close()
                temporary_path.unlink(missing_ok=True)
                raise

        try:
            if temporary_path.stat().st_size != expected_size:
                raise EOFError(f"Incomplete archive member for: {destination}")

            if destination.exists():
                if not destination.is_file():
                    raise IsADirectoryError(str(destination))
                if filecmp.cmp(temporary_path, destination, shallow=False):
                    return False
                quarantined = quarantine_dataset_file(
                    destination, raw_root, quarantine_dir
                )
                replacement_records.append({
                    "archive_path": str(archive_path),
                    "image_path": str(destination),
                    "quarantine_path": str(quarantined),
                    "reason": "existing_bytes_differ_from_archive",
                })

            temporary_path.replace(destination)
            return True
        finally:
            # This is only the newly created temporary extraction output.
            temporary_path.unlink(missing_ok=True)

    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or stat.S_ISLNK(member.external_attr >> 16):
                    continue
                class_name, destination = image_destination(member.filename)
                if destination is None:
                    continue
                with archive.open(member, "r") as source:
                    recovered_counts[class_name] += int(
                        publish_member(source, destination, member.file_size)
                    )
    else:
        with tarfile.open(archive_path, "r:*") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue
                class_name, destination = image_destination(member.name)
                if destination is None:
                    continue
                source = archive.extractfile(member)
                if source is None:
                    continue
                with source:
                    recovered_counts[class_name] += int(
                        publish_member(source, destination, member.size)
                    )

    return recovered_counts


# ------------------------------------------------------------------
# Archive integrity and remote-source inventory
# ------------------------------------------------------------------

ARCHIVE_SOURCE_COLUMNS = [
    "source_id", "source_url", "source_relative_path", "local_path", "filename"
]
ARCHIVE_REPAIR_COLUMNS = [
    "archive_path", "source_url", "initial_problem", "final_status",
    "relevant_classes", "relevant_class_count", "failure_category",
    "download_attempts", "replacement_downloaded", "recovered_images",
    "quarantine_path", "candidate_path", "error_type", "error_message",
]
IMAGE_REPLACEMENT_COLUMNS = [
    "archive_path", "image_path", "quarantine_path", "reason"
]


def archive_error_category(error):
    """Only integrity/read failures can trigger archive replacement."""
    if isinstance(error, (tarfile.CompressionError, NotImplementedError)):
        return "unsupported_compression"
    if isinstance(error, (
        zipfile.BadZipFile, tarfile.ReadError, tarfile.HeaderError,
        gzip.BadGzipFile, EOFError, zlib.error, lzma.LZMAError,
    )):
        return "archive_integrity_error"
    if isinstance(error, OSError):
        return "filesystem_error"
    if isinstance(error, RuntimeError):
        # Includes encrypted ZIP members and missing compression modules.
        return "archive_environment_error"
    return "operation_error"


def validate_archive_integrity(archive_path):
    """Reads archive contents, including ZIP CRCs and compressed TAR trailers."""
    archive_path = Path(archive_path)
    if archive_path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as archive:
            if any(member.flag_bits & 1 for member in archive.infolist()):
                raise RuntimeError("Encrypted ZIP members require a password.")
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f"CRC/header failure in member: {bad_member}")
    elif archive_path.name.lower().endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:*") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue
                source = archive.extractfile(member)
                if source is None:
                    raise EOFError(f"Unreadable TAR member: {member.name}")
                with source:
                    total = sum(len(chunk) for chunk in iter(
                        lambda: source.read(1024 * 1024), b""
                    ))
                if total != member.size:
                    raise EOFError(f"Truncated TAR member: {member.name}")
            # Force compressed-stream EOF/CRC checks even after TAR end markers.
            while archive.fileobj.read(1024 * 1024):
                pass
    else:
        raise ValueError(f"Unsupported archive suffix: {archive_path.name}")


def build_drive_archive_source_manifest(config, dirs):
    """Lists remote archive IDs/paths without downloading their contents."""
    import gdown

    dataset_root = Path(dirs["dataset_root_dir"]).resolve()
    entries = gdown.download_folder(
        url=config["dataset_google_drive_folder_url"],
        output=str(dataset_root),
        quiet=True,
        skip_download=True,
    )
    if entries is None:
        raise RuntimeError("gdown could not list the source Google Drive folder.")

    records = []
    for entry in entries:
        if not hasattr(entry, "id") or not hasattr(entry, "path"):
            raise RuntimeError("Unsupported gdown folder-listing result; update gdown.")
        relative_path = Path(str(entry.path).replace("\\", "/"))
        if not relative_path.name.lower().endswith((".zip", ".tar.gz", ".tgz")):
            continue
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError(f"Unsafe remote archive path: {entry.path}")
        source_id = str(entry.id)
        if not re.fullmatch(r"[A-Za-z0-9_-]+", source_id):
            raise ValueError("gdown returned an invalid Google Drive file ID.")
        local_path = (dataset_root / relative_path).resolve()
        if not local_path.is_relative_to(dataset_root):
            raise ValueError(f"Archive path escapes the dataset directory: {entry.path}")
        records.append({
            "source_id": source_id,
            "source_url": f"https://drive.google.com/uc?id={source_id}",
            "source_relative_path": relative_path.as_posix(),
            "local_path": str(local_path),
            "filename": relative_path.name,
        })

    result = pd.DataFrame.from_records(records, columns=ARCHIVE_SOURCE_COLUMNS)
    if result["source_relative_path"].duplicated().any():
        raise ValueError("Source Drive folder has duplicate archive paths; mapping is ambiguous.")
    return result.sort_values("source_relative_path", kind="stable").reset_index(drop=True)


def match_archive_source(archive_path, source_manifest):
    """Prefers exact local paths; permits a filename match only when unique."""
    archive_path = Path(archive_path).resolve()
    matches = source_manifest.loc[source_manifest["local_path"].eq(str(archive_path))]
    if matches.empty:
        matches = source_manifest.loc[
            source_manifest["filename"].str.casefold().eq(archive_path.name.casefold())
        ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one remote source for {archive_path.name}; found {len(matches)}. "
            "No replacement was downloaded."
        )
    return matches.iloc[0].to_dict()


def quarantine_dataset_file(path, raw_root, quarantine_dir):
    """Moves one exact file to quarantine outside the raw dataset tree."""
    path = Path(path)
    raw_root = Path(raw_root).resolve()
    resolved = path.resolve()
    if path.is_symlink() or not path.is_file() or not resolved.is_relative_to(raw_root):
        raise ValueError(f"Cannot quarantine a non-regular/out-of-dataset path: {path}")
    quarantine_dir = Path(quarantine_dir).resolve()
    if quarantine_dir.is_relative_to(raw_root):
        raise ValueError("Quarantine must be outside the raw-dataset scan root.")
    destination = (
        quarantine_dir / "originals" / uuid4().hex / resolved.relative_to(raw_root)
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    path.rename(destination)
    return destination


def save_provisioning_table(table, path):
    """Publishes a CSV after its temporary write completes."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.part")
    try:
        table.to_csv(temporary, index=False)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def run_labelled_archive_recovery(config, dirs, expected_class_counts):
    """
    Recovers local images, with at most ONE replacement download per
    source archive in this invocation. Counts alone never trigger
    redownload of a present, readable archive.

    Returns report tables, output locations, and operation flags.
    No full-folder redownload is performed by this function.
    """
    raw_root = Path(dirs["raw_data_dir"]).resolve()
    labelled_dir = Path(dirs["labelled_images_dir"]).resolve()
    execution_id = uuid4().hex
    reports_dir = Path(dirs["reports_dir"]) / "archive_repair" / execution_id
    quarantine_dir = raw_root.parent / f"{raw_root.name}_quarantine" / execution_id
    records = []
    replacements = []
    source_manifest = pd.DataFrame(columns=ARCHIVE_SOURCE_COLUMNS)
    source_loaded = False
    source_error = None
    processed_paths = set()
    attempted_source_ids = set()
    recovery_attempted = False
    repair_allowed = bool(
        config["dataset_download_enabled"]
        and config.get("dataset_archive_repair_enabled", True)
    )

    def persist_reports():
        save_provisioning_table(
            pd.DataFrame.from_records(records, columns=ARCHIVE_REPAIR_COLUMNS),
            reports_dir / "archive_repair_report.csv",
        )
        save_provisioning_table(
            pd.DataFrame.from_records(replacements, columns=IMAGE_REPLACEMENT_COLUMNS),
            reports_dir / "image_replacement_report.csv",
        )
        save_provisioning_table(source_manifest, reports_dir / "archive_source_manifest.csv")

    def sources():
        nonlocal source_loaded, source_manifest, source_error
        if not source_loaded:
            source_loaded = True
            try:
                print("Listing source archives in Google Drive (no file download)...")
                source_manifest = build_drive_archive_source_manifest(config, dirs)
            except Exception as error:
                source_error = error
        if source_error is not None:
            raise RuntimeError(f"Archive-source discovery failed: {source_error}") from source_error
        return source_manifest

    def remaining_classes():
        scan = scan_labelled_image_classes(labelled_dir, expected_class_counts)
        return set(scan["missing_classes"]) | set(scan["incomplete_classes"])

    def replace_archive(path, row, known_source):
        if not repair_allowed:
            raise RuntimeError("Archive repair/download is disabled in CONFIG.")
        source = known_source or match_archive_source(path, sources())
        row["source_url"] = source["source_url"]
        if source["source_id"] in attempted_source_ids:
            raise RuntimeError("The single download attempt for this source archive was already used.")
        gdown_executable = shutil.which("gdown")
        if gdown_executable is None:
            raise RuntimeError("gdown is not available on PATH; run %pip install -q gdown.")

        # Stage the replacement outside all raw-dataset/archive scans.
        candidate = quarantine_dir / "downloads" / uuid4().hex / path.name
        candidate.parent.mkdir(parents=True, exist_ok=True)
        row["candidate_path"] = str(candidate)
        row["download_attempts"] = 1
        attempted_source_ids.add(source["source_id"])
        persist_reports()
        print(f"Downloading one archive replacement: {path.name}")
        run_gdown_download([
            gdown_executable, source["source_url"], "-O", str(candidate)
        ])
        if not candidate.is_file():
            raise RuntimeError("gdown returned without creating the replacement archive.")
        validate_archive_integrity(candidate)

        # Do not move/replace the old archive until the new one is verified.
        if path.exists():
            row["quarantine_path"] = str(
                quarantine_dataset_file(path, raw_root, quarantine_dir)
            )
        path.parent.mkdir(parents=True, exist_ok=True)
        candidate.replace(path)
        row["replacement_downloaded"] = True

    def process_archive(path, known_source=None):
        nonlocal recovery_attempted
        path = Path(path).resolve()
        if not path.is_relative_to(raw_root):
            raise ValueError(f"Archive is outside the declared raw-dataset root: {path}")
        if path in processed_paths:
            return
        processed_paths.add(path)
        targets = remaining_classes()
        if not targets:
            return
        row = {
            "archive_path": str(path), "source_url": "", "initial_problem": "",
            "final_status": "in_progress", "download_attempts": 0,
            "relevant_classes": "", "relevant_class_count": 0, "failure_category": "",
            "replacement_downloaded": False, "recovered_images": 0,
            "quarantine_path": "", "candidate_path": "",
            "error_type": "", "error_message": "",
        }
        records.append(row)
        persist_reports()

        try:
            needs_replacement = not path.exists()
            relevant_classes = set()
            if needs_replacement:
                row["initial_problem"] = "missing_archive"
            else:
                try:
                    relevant_classes = inspect_archive_for_classes(path, targets)
                    row["relevant_classes"] = " | ".join(sorted(relevant_classes))
                    row["relevant_class_count"] = len(relevant_classes)
                    if not relevant_classes:
                        row["final_status"] = "not_relevant"
                        return
                    recovery_attempted = True
                    print(f"Checking archive contents: {path.name}")
                    validate_archive_integrity(path)
                except Exception as error:
                    row["initial_problem"] = archive_error_category(error)
                    if row["initial_problem"] != "archive_integrity_error":
                        raise
                    needs_replacement = True

            if needs_replacement:
                recovery_attempted = True
                replace_archive(path, row, known_source)
                relevant_classes = inspect_archive_for_classes(path, targets)
            if not relevant_classes:
                row["final_status"] = "not_relevant"
                return

            row["relevant_classes"] = " | ".join(sorted(relevant_classes))
            row["relevant_class_count"] = len(relevant_classes)
            for extraction_pass in range(2):
                try:
                    print(f"Recovering images from: {path.name}")
                    counts = recover_required_images_from_archive(
                        archive_path=path, labelled_images_dir=labelled_dir,
                        target_classes=relevant_classes, raw_root=raw_root,
                        quarantine_dir=quarantine_dir, replacement_records=replacements,
                    )
                    break
                except Exception as error:
                    category = archive_error_category(error)
                    if (
                        category != "archive_integrity_error"
                        or row["download_attempts"] != 0
                        or extraction_pass != 0
                    ):
                        raise
                    row["initial_problem"] = category
                    replace_archive(path, row, known_source)
            row["recovered_images"] = sum(counts.values())
            row["final_status"] = "recovered" if row["recovered_images"] else "reused"

        except Exception as error:
            row["final_status"] = "failed"
            row["error_type"] = type(error).__name__
            row["error_message"] = str(error)
            row["failure_category"] = archive_error_category(error)
            print(f"Archive operation failed: {path.name}: {error}")
        except BaseException:
            row["final_status"] = "interrupted"
            raise
        finally:
            persist_reports()

    try:
        for archive_path in find_dataset_archives(raw_root):
            if not remaining_classes():
                break
            process_archive(archive_path)
            # Storage/tool/format errors need correction, not more downloads.
            if records and records[-1]["final_status"] == "failed":
                if records[-1]["failure_category"] in {
                    "filesystem_error", "archive_environment_error", "unsupported_compression"
                }:
                    break

        # A missing archive has no local path to discover. Consult the remote
        # inventory for absent class archives / labelled-image bundle archives.
        # A valid archive already present is never downloaded again on counts alone.
        targets = remaining_classes()
        if targets and repair_allowed and not any(row["final_status"] == "failed" for row in records):
            try:
                remote_sources = sources()
                bundle_key = normalize_class_key(Path(dirs["labelled_images_dir"]).name)
                for source in remote_sources.to_dict(orient="records"):
                    targets = remaining_classes()
                    if not targets:
                        break
                    remote_path = Path(source["source_relative_path"])
                    parts = {normalize_class_key(part) for part in remote_path.parts}
                    class_keys = {normalize_class_key(name) for name in targets}
                    relevant = bool(parts & class_keys) or normalize_class_key(remote_path.name) == bundle_key
                    if not relevant:
                        continue
                    expected_path = Path(source["local_path"])
                    # Reuse a uniquely matching archive stored elsewhere in raw.
                    local_matches = [
                        path for path in find_dataset_archives(raw_root)
                        if path.name.casefold() == remote_path.name.casefold()
                    ]
                    if expected_path.exists() or local_matches:
                        continue
                    process_archive(expected_path, known_source=source)
            except Exception as error:
                records.append({
                    "archive_path": "", "source_url": "", "initial_problem": "source_discovery_error",
                    "final_status": "failed", "download_attempts": 0,
                    "relevant_classes": "", "relevant_class_count": 0,
                    "failure_category": "source_discovery_error",
                    "replacement_downloaded": False, "recovered_images": 0,
                    "quarantine_path": "", "candidate_path": "",
                    "error_type": type(error).__name__, "error_message": str(error),
                })
    finally:
        persist_reports()

    return {
        "archive_repair_report": pd.DataFrame.from_records(records, columns=ARCHIVE_REPAIR_COLUMNS),
        "archive_source_manifest": source_manifest,
        "image_replacement_report": pd.DataFrame.from_records(replacements, columns=IMAGE_REPLACEMENT_COLUMNS),
        "archive_recovery_attempted": recovery_attempted,
        "archive_recovery_performed": any(row["recovered_images"] > 0 for row in records),
        "archive_redownload_performed": any(row["replacement_downloaded"] for row in records),
        "operations_succeeded": all(row["final_status"] != "failed" for row in records),
        "reports_dir": reports_dir,
        "quarantine_dir": quarantine_dir,
    }


# ------------------------------------------------------------------
# Ensure presence BEFORE building the initial inventory -- NEW
# ------------------------------------------------------------------

# Clear any successful flag left by a previous notebook execution.
RAW_DATASET_AVAILABLE = False
DOWNLOAD_STATUS = ensure_raw_dataset_present(config=CONFIG, dirs=DIRS)

DATASET_PROVISIONING_METHOD = {
    "already_present": "existing_storage",
    "downloaded": "gdown_google_drive",
}[DOWNLOAD_STATUS]


# ------------------------------------------------------------------
# Initial inventory and class scan
# ------------------------------------------------------------------

RAW_DATASET_INVENTORY = build_raw_dataset_inventory(config=CONFIG, dirs=DIRS)

LABELLED_IMAGE_SCAN = scan_labelled_image_classes(
    labelled_images_dir=labelled_images_dir,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)
labelled_class_report = build_labelled_class_report(
    scan_report=LABELLED_IMAGE_SCAN,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

print("Initial labelled-image inventory:")
print(
    "Classes:",
    f"{LABELLED_IMAGE_SCAN['existing_class_count']}/{EXPECTED_LABELLED_CLASS_COUNT}",
)
print(
    "Images:",
    f"{LABELLED_IMAGE_SCAN['total_actual']:,}/{EXPECTED_LABELLED_IMAGE_COUNT:,}",
)
display(labelled_class_report)


# ------------------------------------------------------------------
# Recover labelled images, repairing damaged/missing source archives
# ------------------------------------------------------------------

ARCHIVE_RECOVERY_RESULT = run_labelled_archive_recovery(
    config=CONFIG,
    dirs=DIRS,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

archive_repair_report = ARCHIVE_RECOVERY_RESULT["archive_repair_report"]
archive_source_manifest = ARCHIVE_RECOVERY_RESULT["archive_source_manifest"]
image_replacement_report = ARCHIVE_RECOVERY_RESULT["image_replacement_report"]
ARCHIVE_RECOVERY_ATTEMPTED = ARCHIVE_RECOVERY_RESULT["archive_recovery_attempted"]
ARCHIVE_RECOVERY_PERFORMED = ARCHIVE_RECOVERY_RESULT["archive_recovery_performed"]
ARCHIVE_REDOWNLOAD_PERFORMED = ARCHIVE_RECOVERY_RESULT["archive_redownload_performed"]
PROVISIONING_REPORTS_DIR = ARCHIVE_RECOVERY_RESULT["reports_dir"]

archive_inventory_report = (
    archive_repair_report.loc[:, [
        "archive_path", "relevant_classes", "relevant_class_count"
    ]]
    .rename(columns={"archive_path": "archive"})
)
display(archive_repair_report)


# ------------------------------------------------------------------
# Final inventory: always rebuild after any download / recovery
# ------------------------------------------------------------------

LABELLED_IMAGE_SCAN = scan_labelled_image_classes(
    labelled_images_dir=labelled_images_dir,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)
labelled_class_report = build_labelled_class_report(
    scan_report=LABELLED_IMAGE_SCAN,
    expected_class_counts=EXPECTED_LABELLED_CLASS_COUNTS,
)

print("\nFinal labelled-image inventory:")
display(labelled_class_report)

RAW_DATASET_INVENTORY = {
    **build_raw_dataset_inventory(config=CONFIG, dirs=DIRS),
    "labelled_class_count": LABELLED_IMAGE_SCAN["existing_class_count"],
    "expected_labelled_class_count": EXPECTED_LABELLED_CLASS_COUNT,
    "expected_labelled_class_count_met": (
        LABELLED_IMAGE_SCAN["existing_class_count"] == EXPECTED_LABELLED_CLASS_COUNT
    ),
    "expected_labelled_classes_present": LABELLED_IMAGE_SCAN[
        "expected_classes_present"
    ],
    "expected_class_image_counts_met": LABELLED_IMAGE_SCAN[
        "expected_class_counts_met"
    ],
    "expected_labelled_image_count_met": LABELLED_IMAGE_SCAN[
        "expected_total_count_met"
    ],
    "missing_labelled_classes": LABELLED_IMAGE_SCAN["missing_classes"],
    "incomplete_labelled_classes": LABELLED_IMAGE_SCAN["incomplete_classes"],
    "excess_labelled_classes": LABELLED_IMAGE_SCAN["excess_classes"],
    "unexpected_labelled_classes": LABELLED_IMAGE_SCAN["unexpected_classes"],
    "archive_recovery_attempted": ARCHIVE_RECOVERY_ATTEMPTED,
    "archive_recovery_performed": ARCHIVE_RECOVERY_PERFORMED,
    "archive_redownload_performed": ARCHIVE_REDOWNLOAD_PERFORMED,
    "archive_recovery_operations_succeeded": ARCHIVE_RECOVERY_RESULT["operations_succeeded"],
    "dataset_provisioning_method": DATASET_PROVISIONING_METHOD,
    "dataset_download_status": DOWNLOAD_STATUS,
    "dataset_download_performed": DOWNLOAD_STATUS == "downloaded",
}

RAW_DATASET_CHECK_COLUMNS = [
    "raw_data_dir_exists",
    "dataset_root_dir_exists",
    "labelled_images_dir_exists",
    "metadata_available",
    "minimum_video_count_met",
    "minimum_labelled_image_count_met",
    "expected_labelled_class_count_met",
    "expected_labelled_classes_present",
    "expected_class_image_counts_met",
    "expected_labelled_image_count_met",
    "archive_recovery_operations_succeeded",
]

raw_dataset_inventory_report = pd.DataFrame.from_records([RAW_DATASET_INVENTORY])

raw_dataset_validation_report = (
    raw_dataset_inventory_report[RAW_DATASET_CHECK_COLUMNS]
    .T
    .rename(columns={0: "check_passed"})
    .rename_axis("validation_check")
    .reset_index()
    .assign(check_passed=lambda data: data["check_passed"].astype("boolean"))
)
failed_dataset_checks = (
    raw_dataset_validation_report.loc[
        ~raw_dataset_validation_report["check_passed"].fillna(False)
    ]
    .reset_index(drop=True)
)
RAW_DATASET_AVAILABLE = bool(
    raw_dataset_validation_report["check_passed"].fillna(False).all()
)

# Keep the aggregate flag consistent with ALL final checks.
RAW_DATASET_INVENTORY = {
    **RAW_DATASET_INVENTORY,
    "validation_passed": RAW_DATASET_AVAILABLE,
}
raw_dataset_inventory_report = raw_dataset_inventory_report.assign(
    validation_passed=RAW_DATASET_AVAILABLE
)

display(raw_dataset_inventory_report.T.rename(columns={0: "value"}))
display(raw_dataset_validation_report)

# Save the final diagnostics BEFORE the validation gate, including failures.
save_provisioning_table(
    labelled_class_report, PROVISIONING_REPORTS_DIR / "labelled_class_report.csv"
)
save_provisioning_table(
    raw_dataset_validation_report, PROVISIONING_REPORTS_DIR / "raw_dataset_validation_report.csv"
)
save_provisioning_table(
    raw_dataset_inventory_report, PROVISIONING_REPORTS_DIR / "raw_dataset_inventory_report.csv"
)
print("Provisioning reports saved to:", PROVISIONING_REPORTS_DIR)

if not RAW_DATASET_AVAILABLE:
    failed_check_names = failed_dataset_checks["validation_check"].tolist()
    raise RuntimeError(
        "Kvasir-Capsule did not pass final content validation "
        "after optional download and archive recovery. "
        f"Provisioning: {DATASET_PROVISIONING_METHOD}. "
        f"Failed checks: {failed_check_names}. "
        f"Missing labelled classes: {LABELLED_IMAGE_SCAN['missing_classes']}. "
        "Incomplete labelled classes: "
        f"{list(LABELLED_IMAGE_SCAN['incomplete_classes'])}. "
        f"Excess labelled classes: {list(LABELLED_IMAGE_SCAN['excess_classes'])}. "
        f"Inspect the reports and dataset contents at: {dataset_root_dir}. "
        "A count mismatch is a validation finding, not proof that another "
        "full download is required. "
        f"Reports: {PROVISIONING_REPORTS_DIR}"
    )

print("Raw dataset provisioning:", DATASET_PROVISIONING_METHOD)
print("Download status:", DOWNLOAD_STATUS)
print("Archive recovery attempted:", ARCHIVE_RECOVERY_ATTEMPTED)
print("Archive recovery performed:", ARCHIVE_RECOVERY_PERFORMED)
print("Archive redownload performed:", ARCHIVE_REDOWNLOAD_PERFORMED)
print("Raw dataset available:", RAW_DATASET_AVAILABLE)
print("Discovered videos:", f"{RAW_DATASET_INVENTORY['video_count']:,}")
print(
    "Discovered labelled classes:",
    f"{LABELLED_IMAGE_SCAN['existing_class_count']}/{EXPECTED_LABELLED_CLASS_COUNT}",
)
print(
    "Discovered labelled images:",
    f"{LABELLED_IMAGE_SCAN['total_actual']:,}/{EXPECTED_LABELLED_IMAGE_COUNT:,}",
)
print("Metadata path:", DIRS["metadata_path"])
